# Stackformer: BERT Masked Language Model Pretraining

This tutorial demonstrates configuring and training a bidirectional BERT encoder on a Masked Language Modeling (MLM) task using `Trainer`.

In [ ]:
import shutil
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from stackformer.models import BERT
from stackformer.training import Trainer, TrainConfig

# 1. Model Definition: BERT Bidirectional Encoder
model = BERT(
    vocab_size=500,
    num_layers=2,
    embed_dim=64,
    num_heads=4,
    seq_len=32,
    hidden_dim=128,
    dropout=0.1,
)
print(f"BERT model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Prepare Synthetic Token Data

In [ ]:
torch.manual_seed(42)
# Synthetic non-padding tokens (1 to 499)
train_x = torch.randint(1, 500, (32, 32))
train_y = torch.randint(1, 500, (32, 32))
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=8, shuffle=True)

val_x = torch.randint(1, 500, (16, 32))
val_y = torch.randint(1, 500, (16, 32))
val_loader = DataLoader(TensorDataset(val_x, val_y), batch_size=8, shuffle=False)

## 3. Train with Trainer

In [ ]:
def compute_loss(logits, targets):
    return F.cross_entropy(logits.view(-1, 500), targets.view(-1))

config = TrainConfig(
    max_epochs=2,
    lr=1e-3,
    optimizer="adamw",
    scheduler="cosine",
    eval_every_n_epochs=1,
    checkpoint_dir="checkpoints_bert",
    save_best=True,
    display="plain",
)

trainer = Trainer(
    model=model,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
    compute_loss=compute_loss,
)

state = trainer.train()
print(f"BERT training completed! Global step: {state.global_step}")

## 4. Save and Reload Model Weights

In [ ]:
trainer.save("bert_final.safetensors")
fresh_model = BERT(
    vocab_size=500,
    num_layers=2,
    embed_dim=64,
    num_heads=4,
    seq_len=32,
    hidden_dim=128,
    dropout=0.1,
)
fresh_model.load_weights("bert_final.safetensors")
print("BERT weights saved via Trainer to safetensors and reloaded successfully!")

# Clean up generated files
import os
if os.path.exists("bert_final.safetensors"):
    os.remove("bert_final.safetensors")
if os.path.exists("checkpoints_bert"):
    shutil.rmtree("checkpoints_bert")